# ## 1. Variables y Gestión de Tipos en Contextos Embebidos
# En los sistemas de IoT, la captura de sensores genera datos con requerimientos
# específicos de precisión y rango. Analizaremos cómo se comportan las variables
# en memoria y cómo validar su tipo.

In [5]:
# Importación de librerías para manejo de tipos y constantes
import sys

# Definición de variables de telemetría de un nodo sensor embebido
identificador_nodo: str = "ESP32_SN_0104"  # Cadena identificadora
temperatura_ambiente: float = 24.5          # Grados Celsius
humedad_relativa: float = 58.0              # Porcentaje
estado_actuador_rele: bool = False          # Estado lógico del relé
lectura_analogica_adc: int = 2048           # Valor ADC de 12 bits (0-4095)

# Verificación de asignación y tipos
print(f"[INFO] Nodo ID: {identificador_nodo} | Tipo: {type(identificador_nodo)}")
print(f"[INFO] Lectura ADC: {lectura_analogica_adc} | Ocupación en memoria: {sys.getsizeof(lectura_analogica_adc)} bytes")

[INFO] Nodo ID: ESP32_SN_0104 | Tipo: <class 'str'>
[INFO] Lectura ADC: 2048 | Ocupación en memoria: 28 bytes


# ## 2. Estructuras de Control Condicional: Toma de Decisiones y Umbrales
# Implementación de un sistema de alerta por sobrecalentamiento en un entorno industrial.

In [6]:
UMBRAL_CRITICO_TEMP: float = 45.0
UMBRAL_ADVERTENCIA_TEMP: float = 35.0

def evaluar_estado_sensor(temp: float) -> str:
    """
    Evalúa la temperatura recibida y determina el estado operacional del nodo.
    Aplica buenas prácticas de Clean Code con guardas de seguridad.
    """
    if temp >= UMBRAL_CRITICO_TEMP:
        return "CRITICO: Desconexión de emergencia activada"
    elif temp >= UMBRAL_ADVERTENCIA_TEMP:
        return "ADVERTENCIA: Activar refrigeración auxiliar"
    else:
        return "NORMAL: Operación dentro de parámetros nominales"

# Prueba de escritorio con simulación de valores
lecturas_simuladas = [22.0, 38.5, 48.2]
for lectura in lecturas_simuladas:
    estado = evaluar_estado_sensor(lectura)
    print(f"Lectura: {lectura}°C -> Resultado: {estado}")

Lectura: 22.0°C -> Resultado: NORMAL: Operación dentro de parámetros nominales
Lectura: 38.5°C -> Resultado: ADVERTENCIA: Activar refrigeración auxiliar
Lectura: 48.2°C -> Resultado: CRITICO: Desconexión de emergencia activada


## 3. Estructuras de Control Iterativas y Validación de Telemetría
# Procesamiento de una ráfaga de datos de sensores simulados mediante bucles for y while.

In [7]:
# Procesamiento con bucle 'for' para cálculo de promedios
lecturas_buffer: list[float] = [23.1, 23.4, 23.0, 24.1, 23.8, 25.0]
suma_lecturas: float = 0.0

for temp in lecturas_buffer:
    suma_lecturas += temp

promedio_temperatura = suma_lecturas / len(lecturas_buffer)
print(f"[METRICAS] Temperatura promedio del buffer: {promedio_temperatura:.2f}°C")

# Bucle 'while' para validación de entrada de datos simulando un puerto serie
buffer_recepcion = [1024, 2048, -1, 4096, 512, -99] # -1 y -99 son errores de lectura
indice: int = 0

print("\n--- Procesamiento de Buffer con Filtrado de Errores ---")
while indice < len(buffer_recepcion):
    valor_adc = buffer_recepcion[indice]
    indice += 1

    # Guarda de seguridad para datos corruptos
    if valor_adc < 0:
        print(f"[WARNING] Muestra inválida detectada ({valor_adc}). Ignorando...")
        continue
    elif valor_adc > 4095:
        print(f"[ERROR] Desbordamiento de lectura ADC ({valor_adc}). Cancelando..")
        break

    print(f"[OK] Muestra procesada correctamente: {valor_adc}")

[METRICAS] Temperatura promedio del buffer: 23.73°C

--- Procesamiento de Buffer con Filtrado de Errores ---
[OK] Muestra procesada correctamente: 1024
[OK] Muestra procesada correctamente: 2048
[WARNING] Muestra inválida detectada (-1). Ignorando...
[ERROR] Desbordamiento de lectura ADC (4096). Cancelando..
